<a href="https://colab.research.google.com/github/alankusumalan/ai-chatbot-python/blob/main/chatbot_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

%pip install -q --upgrade gradio groq


In [4]:

# CHATBOT AI VERSI 12 - PERBAIKAN MEMORI

import gradio as gr
from groq import Groq
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)


# Mengubah riwayat Gradio menjadi teks
def ambil_teks(content):

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        teks = []

        for blok in content:
            if isinstance(blok, dict):
                if blok.get("type") == "text":
                    teks.append(blok.get("text", ""))

        return "\n".join(teks)

    return ""


# Function chatbot
def jawab_ai(pesan, riwayat):

    if not pesan.strip():
        return "Silakan tulis pertanyaan."

    try:
        konteks = []

        # Membaca riwayat percakapan
        for item in riwayat[-10:]:

            role = item.get("role")
            content = ambil_teks(item.get("content"))

            if role in ["user", "assistant"] and content:
                konteks.append({
                    "role": role,
                    "content": content
                })

        # Memeriksa jumlah pesan sebelumnya
        print("Jumlah pesan riwayat:", len(konteks))

        # Menyiapkan pesan untuk Groq
        messages = [
            {
                "role": "system",
                "content": (
                    "Kamu adalah asisten AI berbahasa Indonesia. "
                    "Jawab dengan ramah, jelas, dan singkat. "
                    "Gunakan informasi dari riwayat percakapan "
                    "untuk memahami pertanyaan lanjutan."
                )
            }
        ] + konteks + [
            {
                "role": "user",
                "content": pesan
            }
        ]

        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            max_completion_tokens=1024
        )

        jawaban = response.choices[0].message.content

        return jawaban or "AI tidak menghasilkan jawaban."

    except Exception as error:
        return (
            "Gagal menghubungi AI. "
            f"Jenis error: {type(error).__name__}"
        )


# Membuat tampilan aplikasi
demo = gr.ChatInterface(
    fn=jawab_ai,
    title="🤖 Chatbot AI Versi 12 - Memory Fixed",
    description="Chatbot Groq dengan memori percakapan.",
    examples=[
        "Apa itu AI?",
        "Jelaskan Python untuk pemula"
    ]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba9d85b6d4aa8260de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
